# Forest cover Predictor 25/26 - Project for Data Science and AI for Business
Authors: Andreea Patarlageanu, Harshita Goyal, Anirudh Sudhir, Benedikt Watzinger

In [ ]:
# We will put all imports here
import pandas as pd
import numpy as np
import os
import ydata_profiling

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

## 1. Exploratory Data Analysis

We start with an exploratoy data analysis in order to better understand the data and later to format it efficiently for good results.

### Loading data, checking features

#### Load data

In [ ]:
file_path = "data/train.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Loaded data in df.")
else:
    print("File does not exist.")

In [ ]:
df.shape

In [ ]:
df.head(5)

We see that we have 15120 and more than 50 features. The dimension is quite large, so we need to inspect these features and later 'eliminate' some of them for dimension reduction.

#### Test file

Let's see if the test file is similar to the train one.

In [ ]:
test_file_path = "data/test-full.csv"
if os.path.exists(test_file_path):
    df_test = pd.read_csv(test_file_path)
    print("Loaded test data in df_test.")

In [ ]:
df_test.shape

In [ ]:
df_test.columns

Let's check easily if indeed both $df$ and $df_{test}$ have the same columns:

In [ ]:
df_cols = set(df.columns) - {"Cover_Type"}
df_test_cols = set(df_test.columns)

print("Same columns?", df_cols == df_test_cols)

#### Inspecting the submission file structure

Let's see hwo the submission file looks like, so we know what is expected:

In [ ]:
target = pd.read_csv("data/full_submission.csv")
print(f"Length of submission example: {len(target)} rows.")

In [ ]:
target.head(5)

Now we know that it is expected from us to "de;iver" a file with 2 columns, the Id and the predicted Cover_Type.

## 2. Data inspection

### Data types

Let's inspect the content of the columns and adjust data types if necessary.

In [ ]:
print(df.columns)
print(df.dtypes)

Great! All of them are integer type.

### Missing values

In [ ]:
df.isnull().sum()

There are no missing values! This is great, no modification needed.

## 3. Feature analysis

### Distribution of features

Let's inspect the distribution of the features, except the target.

In [ ]:
numerical_features = df.select_dtypes(include=["int64"]).columns.tolist()
numerical_features.remove("Cover_Type")
numerical_features.remove("Id")

In [ ]:
binary_features = []
continuous_features = []
trivial_featues = []

for feature in numerical_features:
    data = df[feature]

    # We are interested in some "basic" statistics
    n_unique = data.nunique()
    min_val = data.min()
    max_val = data.max()

    if n_unique == 1:
        trivial_featues.append(feature)
        print(
            f"{feature:45s}: {n_unique:4d} unique values; Range {data.unique()}"
        )

    # How many unique values? Important to see if binary/categorical
    elif n_unique == 2:
        binary_features.append(feature)
        print(
            f"{feature:45s}: {n_unique:4d} unique values; Range {data.sort_values().unique()}"
        )

    else:
        continuous_features.append(feature)
        print(
            f"{feature:45s}: {n_unique:4d} unique values; Range [{min_val:4.0f} : {max_val:4.0f}]"
        )

In [ ]:
binary_features

In [ ]:
trivial_featues

We observe that:

- The 4 types of Wilderness Areas are binary.
- All soil types are binary.
- Soil Type 15 takes only the 0 value, so brings no information. We can remove it.

In [ ]:
df = df.drop(trivial_featues, axis=1)

#### Continuous features

Let's better visualize the continuous features:

In [ ]:
continuous_features = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]

In [ ]:
sns.set_style("whitegrid")

nr_features = len(continuous_features)
n_cols = 3
n_rows = (nr_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for idx, feature in enumerate(continuous_features):

    data_to_plot = df[feature].dropna()

    unique_count = data_to_plot.nunique()

    if unique_count < 30:
        bins = unique_count
        kde = False
    else:
        bins = min(int(np.ceil(np.log2(len(data_to_plot)) + 1)), 50)
        kde = True

    sns.histplot(data=data_to_plot, kde=kde, ax=axes[idx], bins=bins, color="steelblue")

    mean_val = data_to_plot.mean()
    median_val = data_to_plot.median()

    axes[idx].axvline(
        mean_val,
        color="red",
        linestyle="--",
        linewidth=2,
        alpha=0.7,
        label=f"Mean: {mean_val:.1f}",
    )
    axes[idx].axvline(
        median_val,
        color="orange",
        linestyle="--",
        linewidth=2,
        alpha=0.7,
        label=f"Median: {median_val:.1f}",
    )

    axes[idx].set_title(f"{feature}", fontsize=11, fontweight="bold")
    axes[idx].set_xlabel(feature.replace("_", " "), fontsize=9)
    axes[idx].set_ylabel("Frequency", fontsize=9)
    axes[idx].legend(fontsize=7, loc="best")
    axes[idx].grid(axis="y", alpha=0.3)

for idx in range(nr_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle("Continuous Feature Distributions", fontsize=16, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()

We observe that the "Horizontal_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways" and "Horizontal_Distance_To_Fire_Points" are severely right-skewed, having extreme outliers. 

One option would be to **Log transform** the right skewed features, to reduce overfitting on outliers!

#### Wilderness Areas

Let's visualize the Wilderness Areas better. From the description of the data given on Kaggle, we know the following names:

In [ ]:
wilderness_features = [
    "Wilderness_Area1",
    "Wilderness_Area2",
    "Wilderness_Area3",
    "Wilderness_Area4",
]

wilderness_names = {
    "Wilderness_Area1": "Rawah",
    "Wilderness_Area2": "Neota",
    "Wilderness_Area3": "Comanche Peak",
    "Wilderness_Area4": "Cache la Poudre",
}

In [ ]:
# Add reference line for perfect balance
plt.figure(figsize=(10, 5))

wilderness_counts = [df[f].sum() for f in wilderness_features]
wilderness_labels = [wilderness_names[f] for f in wilderness_features]

colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12"]

bars = plt.bar(wilderness_labels, wilderness_counts, color=colors, edgecolor="black")

# Add labels
for bar, count in zip(bars, wilderness_counts):
    height = bar.get_height()
    pct = count / len(df) * 100

    if height < 1000:
        plt.text(
            bar.get_x() + bar.get_width() / 2.0,
            height + 100,
            f"{count}\n({pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )
    else:
        plt.text(
            bar.get_x() + bar.get_width() / 2.0,
            height / 2,
            f"{count}\n({pct:.1f}%)",
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
        )

# Add reference line for perfect balance (25% each)
perfect_balance = len(df) / 4
plt.axhline(
    y=perfect_balance,
    color="red",
    linestyle="--",
    linewidth=2,
    alpha=0.5,
    label="Perfect Balance (25%)",
)
plt.legend()

plt.title("Wilderness Area Distribution (Imbalanced)")
plt.xlabel("Wilderness Area")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, max(wilderness_counts) * 1.1)
plt.tight_layout()
plt.show()

We observe that the Wilderness Area is highly imbalanced, especially for the Neota type, which represents only 3.8%. We added a dotted line which hows the level for a perfect balance. 

However, since Wilderness Area is a feature, we should keep in mind this imbalance when trying to predict the Cover type, but for now we leave it as it is.

### Target variable distribution

Let's seethe distribution of the target variable:

In [ ]:
# Simple check: Are all 7 classes balanced?
df["Cover_Type"].value_counts().sort_index()

In [ ]:
# One simple bar chart with different colors
df["Cover_Type"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(10, 5),
    color=["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c", "#34495e"],
)
plt.title("Cover Type Distribution")
plt.xlabel("Cover Type")
plt.ylabel("Count")
plt.show()

### Feature distribution by Cover Type

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

features = ["Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology"]
colors = ["steelblue", "coral", "lightgreen", "plum"]

for idx, feature in enumerate(features):
    sns.boxplot(data=df, x="Cover_Type", y=feature, ax=axes[idx], color=colors[idx])
    axes[idx].set_title(feature)

plt.tight_layout()
plt.show()

**Comment:**

- Elevation might be a strong predictor because of its non-linear relationship for all the Cover Types.
- Aspect by itself doesn't differentiate much the Cover types, but might be useful in interactions.
- Slope can be a moderate predictor, making some differences between Types 1, 2, 7 and 3, 4, 5, 6.
- Horizontal Distance To Hydrology might be an excellent predictor, clearly separating Type 4 of the rest.

### Correlation with the target

In [ ]:
correlations = df.corr()["Cover_Type"].drop("Cover_Type").sort_values(ascending=False)
print(correlations)

In [ ]:
corr_matrix = df[continuous_features].corr()  # No Cover_Type!

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title("Feature-to-Feature Correlation Matrix")
plt.show()

The correlations coefficients range from -1 to 1 and they have the following meaning:

1. if it is close to -1, then it has perfect negative correlation
2. close to 0 means non-linear relationship 
3. close to 1 means perfect positive correlation

### Feature Engineering Pipeline

According to the previous detailed EDA, I propose the following feature engineering:

1. **Distance_To_Hydrology**: We want to know the actual distance to water, by combining the horizontal and vertical distances. We use the Euclidean distance to find it
2. **Log_Horiz_Hydrology, Log_Horiz_Roadways, Log_Horiz_Fire**: We log transform these distances because they are extremely right skewed. 
3. **Aspect_sin, Aspect_cos**: We need to convert the Aspect degrees in sin and cos, because it is Circular and 0 degrees = 360 degrees, but the model would think they are very far apart.
4. **Hillshade_Mean** = The hillshades of 9am, noon and 3pm all measure sun exposure. Let's average them to capture an overall sun exposure thoughout the day.
5. **Elevation_Wilderness_Area** (4 in total): Since a specific elevation can mean "very high" in one area and "normal/very low" in other areas, we create this feature so the models can better understand area-specific elevation patterns.
6. **Close_To_Water**: We want to know if it is close to water (1) or not (0). We will consider "close" to be within 100m.
7. **High_Altitude**: If elevation is greater than 3200m (1) is considered to be high altitude, otherwise not (0).

In [ ]:
def create_features(df):
    df = df.copy()

    df["Distance_To_Hydrology"] = np.sqrt(
        df["Horizontal_Distance_To_Hydrology"] ** 2
        + df["Vertical_Distance_To_Hydrology"] ** 2
    )

    df["Log_Horiz_Hydrology"] = np.log1p(df["Horizontal_Distance_To_Hydrology"])
    df["Log_Horiz_Roadways"] = np.log1p(df["Horizontal_Distance_To_Roadways"])
    df["Log_Horiz_Fire"] = np.log1p(df["Horizontal_Distance_To_Fire_Points"])

    df["Aspect_sin"] = np.sin(df["Aspect"] * np.pi / 180)
    df["Aspect_cos"] = np.cos(df["Aspect"] * np.pi / 180)

    df["Hillshade_Mean"] = (
        df["Hillshade_9am"] + df["Hillshade_Noon"] + df["Hillshade_3pm"]
    ) / 3

    df[f"Elevation_Wilderness_Area1"] = df["Elevation"] * df[f"Wilderness_Area1"]
    df[f"Elevation_Wilderness_Area2"] = df["Elevation"] * df[f"Wilderness_Area2"]
    df[f"Elevation_Wilderness_Area3"] = df["Elevation"] * df[f"Wilderness_Area3"]
    df[f"Elevation_Wilderness_Area4"] = df["Elevation"] * df[f"Wilderness_Area4"]

    df["Close_To_Water"] = (df["Horizontal_Distance_To_Hydrology"] < 100).astype(int)
    df["High_Altitude"] = (df["Elevation"] > 3200).astype(int)

    return df

We apply these to both train and test data sets, and we will call them from now on **df_train_fe** and **df_test_fe**:

In [ ]:
df_train_fe = create_features(df)
df_test_fe = create_features(df_test)

Now, let's remove the Features that we already integrated in the more complex ones. Like this, we reduce the dimensionality as well.

In [ ]:
to_drop = [
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Horizontal_Distance_To_Fire_Points",
    "Aspect",
    "Elevation",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Wilderness_Area1",
    "Wilderness_Area2",
    "Wilderness_Area3",
    "Wilderness_Area4"
]

In [ ]:
df_train_fe = df_train_fe.drop(columns=to_drop)
df_test_fe = df_test_fe.drop(columns=to_drop)

In [ ]:
df_train_fe.shape, df_test_fe.shape

In [ ]:
df_train_fe.columns

In [ ]:
df.profile_report()